# Section B

# Q4

In [1]:
import random

In [2]:
#  CORE GA COMPONENTS (used across all parts)

def decode(chromosome):
    """
    Convert a 4-bit binary list to an integer x.
    chromosome[0] is the Most Significant Bit (MSB).
    Example: [1, 0, 1, 1] -> 1*8 + 0*4 + 1*2 + 1*1 = 11
    """
    x = 0
    for bit in chromosome:
        x = (x << 1) | bit
    return x

In [3]:
def fitness(chromosome):
    """
    Returns f(x) = -x^2 + 14x + 5 for the given chromosome.
    Decodes first, then evaluates.
    """
    x = decode(chromosome)
    return -(x ** 2) + 14 * x + 5

In [4]:
def roulette_select(population):
    """
    Selects ONE individual using fitness-proportionate (roulette wheel) selection.
    Steps:
      1. Compute raw fitness for each individual.
      2. Shift all values to be strictly positive (add offset if any f <= 0).
      3. Build cumulative probability distribution.
      4. Spin with random.random() and return the individual whose range contains it.
    """
    raw_scores = [fitness(ind) for ind in population]

    min_score = min(raw_scores)
    offset    = abs(min_score) + 1 if min_score <= 0 else 0
    adjusted  = [s + offset for s in raw_scores]

    total      = sum(adjusted)
    spin       = random.random()
    cumulative = 0.0

    for individual, adj_score in zip(population, adjusted):
        cumulative += adj_score / total
        if spin <= cumulative:
            return individual

    return population[-1]   # floating-point safety fallback

In [5]:
def single_point_crossover(parent1, parent2, point):
    """
    Single-point crossover at bit position `point` (after index `point`).
    offspring1 = parent1[:point] + parent2[point:]
    offspring2 = parent2[:point] + parent1[point:]
    point must be in [1, len-1].
    """
    offspring1 = parent1[:point] + parent2[point:]
    offspring2 = parent2[:point] + parent1[point:]
    return offspring1, offspring2

In [6]:
def mutate(chromosome, p_m):
    """
    Per-bit flip mutation. Each bit is independently flipped with probability p_m.
    Returns a NEW list (does not modify in place).
    """
    return [1 - bit if random.random() < p_m else bit
            for bit in chromosome]

In [7]:
def init_population(pop_size, bits=4):
    """Generate a random population of pop_size chromosomes of length `bits`."""
    return [[random.randint(0, 1) for _ in range(bits)] for _ in range(pop_size)]

In [8]:
#  PART (a) - Core Components: isolated tests

def main_part_a():
    print("PART (a) - Core GA Components: decode, fitness, roulette_select,")
    print("           single_point_crossover, mutate")

    print("""
Objective function  : f(x) = -x^2 + 14x + 5,   x in {0, 1, ..., 15}
Chromosome          : 4-bit binary list, MSB first
Global maximum      : x = 7,  f(7) = -(49) + 98 + 5 = 54
Test chromosomes    : [0,1,1,0], [1,0,0,1], [1,1,0,0], [0,0,1,1]
""")

    test_chroms = [
        [0, 1, 1, 0],   # x = 6,  f =  53
        [1, 0, 0, 1],   # x = 9,  f =  50
        [1, 1, 0, 0],   # x = 12, f =  29
        [0, 0, 1, 1],   # x = 3,  f =  38
    ]

    # TEST 1 : decode()
    print("--- TEST 1: decode(chromosome) ---")
    print(f"  Rule: MSB first -> value = b3*8 + b2*4 + b1*2 + b0*1\n")
    print(f"  {'Chromosome':<20}  {'Working':<40}  {'x':>4}")
    print(f"  {'-'*70}")
    expected_x = [6, 9, 12, 3]
    for chrom, exp in zip(test_chroms, expected_x):
        x   = decode(chrom)
        wrk = (f"{chrom[0]}*8 + {chrom[1]}*4 + "
               f"{chrom[2]}*2 + {chrom[3]}*1 = {x}")
        ok  = "OK" if x == exp else f"ERROR (expected {exp})"
        print(f"  {str(chrom):<20}  {wrk:<40}  {x:>4}  {ok}")

    # TEST 2 : fitness()
    print(f"\n--- TEST 2: fitness(chromosome)  [f(x) = -x^2 + 14x + 5] ---\n")
    print(f"  {'Chromosome':<20}  {'x':>4}  {'Working':<40}  {'f(x)':>6}")
    for chrom in test_chroms:
        x   = decode(chrom)
        fv  = fitness(chrom)
        wrk = f"-({x})^2 + 14*{x} + 5 = -{x**2} + {14*x} + 5 = {fv}"
        print(f"  {str(chrom):<20}  {x:>4}  {wrk:<40}  {fv:>6}")

    print(f"\n  Reference: x=7 -> f(7) = -(49) + 98 + 5 = 54  [GLOBAL MAXIMUM]")

    # TEST 3 : roulette_select()
    print(f"\n--- TEST 3: roulette_select(population) ---\n")

    f_vals  = [fitness(c) for c in test_chroms]
    total_f = sum(f_vals)
    print(f"  Population fitness values : {f_vals}")
    print(f"  Total fitness sum         : {total_f}")
    print(f"\n  Selection probabilities (P = f / total_f):")
    print(f"  {'Chromosome':<20}  {'f(x)':>6}  {'f/total':>10}  {'Cumulative':>12}")
    cum = 0.0
    for chrom, fv in zip(test_chroms, f_vals):
        p   = fv / total_f
        cum += p
        print(f"  {str(chrom):<20}  {fv:>6}  {p:>10.4f}  {cum:>12.4f}")

    print(f"\n  Empirical check -- 10,000 spins (seed=42):")
    random.seed(42)
    counts = [0] * len(test_chroms)
    for _ in range(10000):
        sel = roulette_select(test_chroms)
        for i, ind in enumerate(test_chroms):
            if sel == ind:
                counts[i] += 1
                break
    print(f"  {'Chromosome':<20}  {'Theory':>8}  {'Empirical':>10}  {'Match?':>8}")
    print(f"  {'-'*52}")
    for chrom, fv, cnt in zip(test_chroms, f_vals, counts):
        theory = fv / total_f
        emp    = cnt / 10000
        match  = "YES" if abs(theory - emp) < 0.02 else "DEVIATION"
        print(f"  {str(chrom):<20}  {theory:>8.4f}  {emp:>10.4f}  {match:>8}")

    # TEST 4 : single_point_crossover()
    print(f"\n--- TEST 4: single_point_crossover(parent1, parent2, point) ---\n")
    p1, p2 = [0, 1, 1, 0], [1, 0, 0, 1]
    print(f"  Parent1 = {p1}   (x={decode(p1)}, f={fitness(p1)})")
    print(f"  Parent2 = {p2}   (x={decode(p2)}, f={fitness(p2)})\n")
    print(f"  {'Point':<8}  {'Offspring1':<20}  {'x':>4}  {'f':>4}  {'Offspring2':<20}  {'x':>4}  {'f':>4}")
    for pt in [1, 2, 3]:
        o1, o2 = single_point_crossover(p1, p2, pt)
        note1  = " <- global max!" if decode(o1) == 7 else ""
        note2  = " <- global max!" if decode(o2) == 7 else ""
        print(f"  {pt:<8}  {str(o1):<20}  {decode(o1):>4}  {fitness(o1):>4}{note1}"
              f"  {str(o2):<20}  {decode(o2):>4}  {fitness(o2):>4}{note2}")

    # TEST 5 : mutate()
    print(f"\n--- TEST 5: mutate(chromosome, p_m) ---\n")
    base = [0, 1, 1, 0]
    print(f"  Base chromosome: {base}  (x={decode(base)}, f={fitness(base)})\n")
    print(f"  {'p_m':<6}  {'Expected flips/run':<22}  {'Sample results (5 runs, seed=10)'}")
    for p_m in [0.0, 0.1, 0.5, 1.0]:
        random.seed(10)
        samples    = [mutate(base, p_m) for _ in range(5)]
        flip_cnts  = [sum(1 for a, b in zip(base, s) if a != b) for s in samples]
        exp_flips  = f"~{4 * p_m:.1f} bits"
        sample_str = "  ".join(f"{s}({fc}f)" for s, fc in zip(samples, flip_cnts))
        print(f"  {p_m:<6}  {exp_flips:<22}  {sample_str}")

    # SUMMARY TABLE
    print(f"\n--- SUMMARY: All Test Chromosomes ---\n")
    total_f = sum(fitness(c) for c in test_chroms)
    print(f"  {'Individual':<12}  {'Chromosome':<20}  {'x':>4}  {'f(x)':>6}  {'Sel. Prob':>10}")
    for i, chrom in enumerate(test_chroms, 1):
        x  = decode(chrom)
        fv = fitness(chrom)
        sp = fv / total_f
        print(f"  P{i:<11}  {str(chrom):<20}  {x:>4}  {fv:>6}  {sp:>10.4f}")
    print(f"\n  Total fitness = {total_f}")
    print(f"  Target (global max): chromosome [0,1,1,1] -> x=7, f=54")
    print(f"  Note: none of the four test chromosomes encode x=7.")


In [9]:
main_part_a()

PART (a) - Core GA Components: decode, fitness, roulette_select,
           single_point_crossover, mutate

Objective function  : f(x) = -x^2 + 14x + 5,   x in {0, 1, ..., 15}
Chromosome          : 4-bit binary list, MSB first
Global maximum      : x = 7,  f(7) = -(49) + 98 + 5 = 54
Test chromosomes    : [0,1,1,0], [1,0,0,1], [1,1,0,0], [0,0,1,1]

--- TEST 1: decode(chromosome) ---
  Rule: MSB first -> value = b3*8 + b2*4 + b1*2 + b0*1

  Chromosome            Working                                      x
  ----------------------------------------------------------------------
  [0, 1, 1, 0]          0*8 + 1*4 + 1*2 + 0*1 = 6                    6  OK
  [1, 0, 0, 1]          1*8 + 0*4 + 0*2 + 1*1 = 9                    9  OK
  [1, 1, 0, 0]          1*8 + 1*4 + 0*2 + 0*1 = 12                  12  OK
  [0, 0, 1, 1]          0*8 + 0*4 + 1*2 + 1*1 = 3                    3  OK

--- TEST 2: fitness(chromosome)  [f(x) = -x^2 + 14x + 5] ---

  Chromosome               x  Working               

In [16]:
#  PART (b) - Full run_ga() function: generation-by-generation

def run_ga(pop_size, num_generations, p_m, elitism, verbose=True):
    """
    Run a complete Genetic Algorithm.

    Parameters
    ----------
    pop_size        : int   Population size.
    num_generations : int   Number of generations to evolve.
    p_m             : float Per-bit mutation probability.
    elitism         : bool  If True, best individual always passes to next gen.
    verbose         : bool  Print generation table when True.

    Returns
    -------
    history : list of (generation, best_fitness, best_x) tuples.
    """
    population = init_population(pop_size)
    history    = []

    for gen in range(1, num_generations + 1):

        # Evaluate
        scored = [(ind, decode(ind), fitness(ind)) for ind in population]
        scored.sort(key=lambda t: t[2], reverse=True)

        best_ind, best_x, best_fit = scored[0]
        history.append((gen, best_fit, best_x))

        if verbose:
            print(f"\n  Generation {gen}")
            print(f"  {'Rank':<5}  {'Chromosome':<20}  {'x':>4}  {'f(x)':>6}  {'Note'}")
            for rank, (ind, x, f) in enumerate(scored, 1):
                note = ""
                if rank == 1:
                    note = "BEST"
                if x == 7:
                    note += (" | " if note else "") + "GLOBAL MAX"
                print(f"  {rank:<5}  {str(ind):<20}  {x:>4}  {f:>6}  {note}")
            print(f"  Best this generation: x={best_x}, f={best_fit}")

        # Build next generation
        next_pop = []
        if elitism:
            next_pop.append(best_ind[:])   # carry elite unchanged

        while len(next_pop) < pop_size:
            p1    = roulette_select(population)
            p2    = roulette_select(population)
            point = random.randint(1, 3)   # random crossover point in [1,3]
            o1, o2 = single_point_crossover(p1, p2, point)
            o1 = mutate(o1, p_m)
            o2 = mutate(o2, p_m)
            next_pop.append(o1)
            if len(next_pop) < pop_size:
                next_pop.append(o2)

        population = next_pop

    return history

In [17]:
def main_part_b():
    print("PART (b) - run_ga(): Generation-by-Generation Trace")
    print("           pop_size=4, num_generations=10, p_m=0.1, elitism=False")

    print("""
Algorithm overview:
  1. Initialise a random population of pop_size=4 chromosomes (4 bits each).
  2. Each generation:
       a) Evaluate fitness of every individual.
       b) Select two parents via roulette wheel selection.
       c) Apply single-point crossover at a random bit position.
       d) Apply per-bit mutation at rate p_m=0.1.
       e) Replace the old population with the new offspring.
  3. Track the best individual per generation.
  Elitism=False means the best individual is NOT guaranteed to survive.
""")

    random.seed(42)
    history = run_ga(pop_size=4, num_generations=10, p_m=0.1,
                     elitism=False, verbose=True)

    print("\n  Convergence Summary ")
    print(f"\n  {'Gen':>4}  {'Best x':>7}  {'Best f(x)':>10}  {'Progress'}")
    prev_best = None
    for gen, bf, bx in history:
        arrow = " ^ improved" if (prev_best is None or bf > prev_best) else ""
        gm    = " [GLOBAL MAX]" if bx == 7 else ""
        print(f"  {gen:>4}  {bx:>7}  {bf:>10}{arrow}{gm}")
        prev_best = bf

    max_fit_found = max(bf for _, bf, _ in history)
    found_optimum = any(bx == 7 for _, _, bx in history)
    print(f"\n  Global optimum (x=7, f=54) found : {'YES' if found_optimum else 'NO'}")
    print(f"  Maximum fitness reached          : {max_fit_found}")

In [18]:
main_part_b()

PART (b) - run_ga(): Generation-by-Generation Trace
           pop_size=4, num_generations=10, p_m=0.1, elitism=False

Algorithm overview:
  1. Initialise a random population of pop_size=4 chromosomes (4 bits each).
  2. Each generation:
       a) Evaluate fitness of every individual.
       b) Select two parents via roulette wheel selection.
       c) Apply single-point crossover at a random bit position.
       d) Apply per-bit mutation at rate p_m=0.1.
       e) Replace the old population with the new offspring.
  3. Track the best individual per generation.
  Elitism=False means the best individual is NOT guaranteed to survive.


  Generation 1
  Rank   Chromosome               x    f(x)  Note
  1      [1, 0, 0, 0]             8      53  BEST
  2      [0, 0, 1, 0]             2      29  
  3      [0, 0, 0, 0]             0       5  
  4      [0, 0, 0, 0]             0       5  
  Best this generation: x=8, f=53

  Generation 2
  Rank   Chromosome               x    f(x)  Note
  1  

In [36]:
print("""
Analysis:

  Population structure (pop_size=4):
    With only 4 individuals, diversity is extremely limited. Each chromosome
    covers one of 16 possible x values, so 4 individuals represent just 25%
    of the search space per generation. Despite this, roulette selection
    naturally steers offspring toward higher-fitness individuals.

  Role of crossover:
    Single-point crossover at a random position [1,2,3] recombines bit patterns
    from two parents. In a 4-bit space this is powerful -- crossover at point=3
    between [0,1,1,0] and [1,0,0,1] produces [0,1,1,1] which encodes x=7,
    the global maximum. The GA assembles good bit-blocks (schemata) from
    different parents, gradually improving the population.

  Role of mutation at p_m=0.1:
    Expected flips per chromosome = 4 * 0.1 = 0.4, meaning most chromosomes
    are unchanged by mutation each generation. This is deliberate: low mutation
    maintains the diversity generated by crossover without disrupting solutions
    already found. Mutation's main role is to re-introduce diversity if the
    population converges prematurely.

  Without elitism, regression can occur:
    A good individual found in generation k can be lost in generation k+1 if
    roulette selection does not pick it as a parent. This is visible in the
    convergence summary above as "non-monotone" fitness curves. Elitism, tested
    in Part (c), guarantees the best solution is always preserved.
""")


Analysis:

  Population structure (pop_size=4):
    With only 4 individuals, diversity is extremely limited. Each chromosome
    covers one of 16 possible x values, so 4 individuals represent just 25%
    of the search space per generation. Despite this, roulette selection
    naturally steers offspring toward higher-fitness individuals.

  Role of crossover:
    Single-point crossover at a random position [1,2,3] recombines bit patterns
    from two parents. In a 4-bit space this is powerful -- crossover at point=3
    between [0,1,1,0] and [1,0,0,1] produces [0,1,1,1] which encodes x=7,
    the global maximum. The GA assembles good bit-blocks (schemata) from
    different parents, gradually improving the population.

  Role of mutation at p_m=0.1:
    Expected flips per chromosome = 4 * 0.1 = 0.4, meaning most chromosomes
    are unchanged by mutation each generation. This is deliberate: low mutation
    maintains the diversity generated by crossover without disrupting solutions
   

In [37]:
#  PART (c) - Controlled Experiments

def main_part_c_experiment1():
    print("PART (c) - EXPERIMENT 1: Elitism=False vs Elitism=True")
    print("           pop_size=4, num_generations=20, p_m=0.1, 30 trials")

    print("""
What we measure per trial:
  - Best fitness reached across all 20 generations.
  - Whether x=7 (f=54, the global optimum) was ever found.
  - The generation number where f >= 50 was FIRST achieved (or Never).
""")

    POP_SIZE = 4
    NUM_GENS = 20
    P_M      = 0.1
    TRIALS   = 30

    def run_condition(elitism, seed_offset):
        results = []
        for t in range(TRIALS):
            random.seed(seed_offset + t)
            history  = run_ga(POP_SIZE, NUM_GENS, P_M, elitism, verbose=False)
            best_f   = max(bf for _, bf, _ in history)
            found    = any(bx == 7 for _, _, bx in history)
            gen_50   = next((gen for gen, bf, _ in history if bf >= 50), None)
            results.append((best_f, found, gen_50))
        return results

    print("[c.1] Running Elitism=False (30 trials, seeds 0-29)...")
    res_no  = run_condition(elitism=False, seed_offset=0)

    print("[c.2] Running Elitism=True  (30 trials, seeds 1000-1029)...")
    res_yes = run_condition(elitism=True,  seed_offset=1000)

    # Per-trial detail tables
    for label, results in [("Elitism=False", res_no), ("Elitism=True", res_yes)]:
        print(f"\n {label}: Per-Trial Results")
        print(f"  {'Trial':>6}  {'Best f':>7}  {'Found x=7?':>11}  {'First gen f>=50':>16}")
        for t, (bf, found, g50) in enumerate(results, 1):
            g_str   = f"Gen {g50}" if g50 is not None else "Never"
            opt_str = "YES" if found else "No"
            print(f"  {t:>6}  {bf:>7}  {opt_str:>11}  {g_str:>16}")

    # Aggregate statistics
    def aggregate(results):
        avg_f    = sum(bf for bf, _, _ in results) / TRIALS
        n_opt    = sum(1 for _, found, _ in results if found)
        gens_hit = [g for _, _, g in results if g is not None]
        avg_g    = sum(gens_hit) / len(gens_hit) if gens_hit else None
        n_hit    = len(gens_hit)
        return avg_f, n_opt, avg_g, n_hit

    avg_f_no,  n_opt_no,  avg_g_no,  ng_no  = aggregate(res_no)
    avg_f_yes, n_opt_yes, avg_g_yes, ng_yes = aggregate(res_yes)

    print(f"\n Aggregate Comparison Table \n")
    print(f"  {'Metric':<40}  {'Elitism=False':>14}  {'Elitism=True':>13}")
    print(f"  {'Average best fitness (30 trials)':<40}  {avg_f_no:>14.2f}  {avg_f_yes:>13.2f}")
    print(f"  {'Runs that found x=7 (f=54)':<40}  {str(n_opt_no)+'/30':>14}  {str(n_opt_yes)+'/30':>13}")

    g_no_str  = f"{avg_g_no:.2f} ({ng_no}/30 runs)"  if avg_g_no  else "N/A"
    g_yes_str = f"{avg_g_yes:.2f} ({ng_yes}/30 runs)" if avg_g_yes else "N/A"
    print(f"  {'Avg gen to first f>=50':<40}  {g_no_str:>14}  {g_yes_str:>13}")
    print(f"  {'Min best fitness across trials':<40}  {min(bf for bf,_,_ in res_no):>14}  {min(bf for bf,_,_ in res_yes):>13}")
    print(f"  {'Max best fitness across trials':<40}  {max(bf for bf,_,_ in res_no):>14}  {max(bf for bf,_,_ in res_yes):>13}")

    # Per-generation average best fitness
    print(f"\n Per-Generation Avg Best Fitness (averaged over 30 trials) \n")
    print(f"  {'Gen':>4}  {'Avg f (No Elitism)':>20}  {'Avg f (Elitism)':>17}")
    all_no  = [run_ga(POP_SIZE, NUM_GENS, P_M, False, verbose=False)
               for _ in [random.seed(i) or i for i in range(TRIALS)]]
    all_yes = [run_ga(POP_SIZE, NUM_GENS, P_M, True,  verbose=False)
               for _ in [random.seed(1000+i) or i for i in range(TRIALS)]]
    for g_idx in range(NUM_GENS):
        avg_no  = sum(all_no [t][g_idx][1] for t in range(TRIALS)) / TRIALS
        avg_yes = sum(all_yes[t][g_idx][1] for t in range(TRIALS)) / TRIALS
        print(f"  {g_idx+1:>4}  {avg_no:>20.2f}  {avg_yes:>17.2f}")

In [38]:
main_part_c_experiment1()

PART (c) - EXPERIMENT 1: Elitism=False vs Elitism=True
           pop_size=4, num_generations=20, p_m=0.1, 30 trials

What we measure per trial:
  - Best fitness reached across all 20 generations.
  - Whether x=7 (f=54, the global optimum) was ever found.
  - The generation number where f >= 50 was FIRST achieved (or Never).

[c.1] Running Elitism=False (30 trials, seeds 0-29)...
[c.2] Running Elitism=True  (30 trials, seeds 1000-1029)...

 Elitism=False: Per-Trial Results
   Trial   Best f   Found x=7?   First gen f>=50
       1       54          YES             Gen 1
       2       54          YES            Gen 11
       3       54          YES             Gen 1
       4       54          YES             Gen 1
       5       54          YES             Gen 1
       6       54          YES             Gen 2
       7       54          YES             Gen 1
       8       53           No             Gen 1
       9       54          YES             Gen 1
      10       54          YES  

In [39]:
def main_part_c_experiment2():
    print("PART (c) - EXPERIMENT 2: Mutation Rate Comparison")
    print("           pop_size=4, num_generations=20, elitism=False, 30 trials")

    print("""
What we measure per trial:
  - Best fitness reached across all 20 generations.
  - Whether x=7 (f=54) was found.
  - Average generation where f >= 50 was first achieved.
Mutation rates tested: p_m in {0.01, 0.1, 0.3, 0.5}
""")

    POP_SIZE = 4
    NUM_GENS = 20
    TRIALS   = 30
    PM_LIST  = [0.01, 0.1, 0.3, 0.5]

    agg_data = {}
    all_histories = {}

    for pm_idx, p_m in enumerate(PM_LIST):
        best_fs, found_opts, gen_50s = [], [], []
        histories = []

        for t in range(TRIALS):
            random.seed(pm_idx * 100 + t)
            history = run_ga(POP_SIZE, NUM_GENS, p_m, elitism=False, verbose=False)
            histories.append(history)

            best_f = max(bf for _, bf, _ in history)
            found  = any(bx == 7 for _, _, bx in history)
            gen_50 = next((gen for gen, bf, _ in history if bf >= 50), None)

            best_fs.append(best_f)
            found_opts.append(found)
            gen_50s.append(gen_50)

        all_histories[p_m] = histories

        gens_hit = [g for g in gen_50s if g is not None]
        agg_data[p_m] = {
            'avg_f'  : sum(best_fs) / TRIALS,
            'n_opt'  : sum(found_opts),
            'avg_g50': sum(gens_hit) / len(gens_hit) if gens_hit else None,
            'n_g50'  : len(gens_hit),
            'min_f'  : min(best_fs),
            'max_f'  : max(best_fs),
            'best_fs': best_fs,
        }

    # Per-rate detail tables
    for p_m in PM_LIST:
        histories = all_histories[p_m]
        print(f"\n  p_m={p_m}: Per-Trial Results ")
        print(f"  {'Trial':>6}  {'Best f':>7}  {'Found x=7?':>11}  {'First gen f>=50':>16}")
        print(f"  {'-'*47}")
        for t in range(TRIALS):
            random.seed(PM_LIST.index(p_m) * 100 + t)
            history = all_histories[p_m][t]
            bf      = max(bff for _, bff, _ in history)
            found   = any(bx == 7 for _, _, bx in history)
            g50     = next((gen for gen, bff, _ in history if bff >= 50), None)
            g_str   = f"Gen {g50}" if g50 is not None else "Never"
            o_str   = "YES" if found else "No"
            print(f"  {t+1:>6}  {bf:>7}  {o_str:>11}  {g_str:>16}")

    # Aggregate comparison table
    print(f"\n  Aggregate Comparison Table \n")
    print(f"  {'p_m':>6}  {'Avg Best f':>11}  {'Found x=7':>11}  "
          f"{'Avg Gen f>=50':>15}  {'Min f':>7}  {'Max f':>7}")
    print(f"  {'-'*66}")
    for p_m in PM_LIST:
        d    = agg_data[p_m]
        g_str= f"{d['avg_g50']:.1f} ({d['n_g50']}/30)" if d['avg_g50'] else "N/A"
        print(f"  {p_m:>6}  {d['avg_f']:>11.2f}  {str(d['n_opt'])+'/30':>11}  "
              f"{g_str:>15}  {d['min_f']:>7}  {d['max_f']:>7}")

    # Fitness distribution
    print(f"\n  --- Best Fitness Distribution Across 30 Trials ---\n")
    print(f"  {'f value':>8}", end="")
    for p_m in PM_LIST:
        print(f"  {'p_m='+str(p_m):>10}", end="")
    print()
    print(f"  {'-'*(10 + 14 * len(PM_LIST))}")
    for fv in [54, 53, 50, 45, 38, 29]:
        print(f"  {fv:>8}", end="")
        for p_m in PM_LIST:
            cnt = sum(1 for bf in agg_data[p_m]['best_fs'] if bf == fv)
            print(f"  {cnt if cnt > 0 else '-':>10}", end="")
        print()

    # Per-generation trace
    print(f"\n  Avg Best Fitness Per Generation (30-trial avg) \n")
    print(f"  {'Gen':>4}", end="")
    for p_m in PM_LIST:
        print(f"  {'p_m='+str(p_m):>12}", end="")
    print()
    print(f"  {'-'*(6 + 14 * len(PM_LIST))}")
    for g_idx in range(NUM_GENS):
        print(f"  {g_idx+1:>4}", end="")
        for p_m in PM_LIST:
            avg = sum(all_histories[p_m][t][g_idx][1] for t in range(TRIALS)) / TRIALS
            print(f"  {avg:>12.2f}", end="")
        print()

    best_pm  = max(PM_LIST, key=lambda pm: agg_data[pm]['avg_f'])
    worst_pm = min(PM_LIST, key=lambda pm: agg_data[pm]['avg_f'])

In [40]:
main_part_c_experiment2()

PART (c) - EXPERIMENT 2: Mutation Rate Comparison
           pop_size=4, num_generations=20, elitism=False, 30 trials

What we measure per trial:
  - Best fitness reached across all 20 generations.
  - Whether x=7 (f=54) was found.
  - Average generation where f >= 50 was first achieved.
Mutation rates tested: p_m in {0.01, 0.1, 0.3, 0.5}


  p_m=0.01: Per-Trial Results 
   Trial   Best f   Found x=7?   First gen f>=50
  -----------------------------------------------
       1       53           No             Gen 1
       2       38           No             Never
       3       54          YES             Gen 1
       4       53           No             Gen 1
       5       54          YES             Gen 1
       6       53           No             Gen 2
       7       54          YES             Gen 1
       8       53           No             Gen 1
       9       53           No             Gen 1
      10       54          YES             Gen 1
      11       54          YES       

In [42]:
def main_part_d():
    print("PART (d) - Manual Trace: One Complete GA Generation")
    print("           No code-based GA calls -- all arithmetic shown explicitly")

    print("""
Starting population and given random numbers:
  P1 = [0,1,1,0]    r1 = 0.12  (selection spin 1)
  P2 = [1,0,0,1]    r2 = 0.47  (selection spin 2)
  P3 = [1,1,0,0]    r3 = 0.68  (selection spin 3)
  P4 = [0,0,1,1]    r4 = 0.91  (selection spin 4)
  Crossover point = 2 (after bit index 2)
  Mutation on O1 : per-bit random values = [0.08, 0.43, 0.91, 0.05], p_m = 0.1
""")

    pop_raw = {
        'P1': [0, 1, 1, 0],
        'P2': [1, 0, 0, 1],
        'P3': [1, 1, 0, 0],
        'P4': [0, 0, 1, 1],
    }

    # STEP 1 : Decode and compute f(x)
    print(" STEP 1: Decode chromosomes and compute f(x) = -x^2 + 14x + 5 \n")

    decoded = {}
    for name, chrom in pop_raw.items():
        x  = chrom[0]*8 + chrom[1]*4 + chrom[2]*2 + chrom[3]*1
        fv = -(x**2) + 14*x + 5
        decoded[name] = (chrom, x, fv)
        wrk_x  = f"{chrom[0]}*8 + {chrom[1]}*4 + {chrom[2]}*2 + {chrom[3]}*1 = {x}"
        wrk_f  = f"-{x}^2 + 14*{x} + 5 = -{x**2} + {14*x} + 5 = {fv}"
        print(f"  {name} = {chrom}")
        print(f"       decode  : {wrk_x}")
        print(f"       f(x)    : {wrk_f}")

    total_f = sum(d[2] for d in decoded.values())
    vals    = list(decoded.values())
    print(f"\n  Total fitness Sf = {vals[0][2]} + {vals[1][2]} + "
          f"{vals[2][2]} + {vals[3][2]} = {total_f}")

    # STEP 2 : Selection probabilities
    print(f"\n STEP 2: Selection probabilities and cumulative ranges \n")
    print(f"  P(select) = f(x) / Sf\n")
    print(f"  {'Individual':<12}  {'Chromosome':<16}  {'x':>4}  {'f(x)':>6}  "
          f"{'P(sel)':>8}  {'Cumulative':>12}")

    cum       = 0.0
    cum_data  = {}
    for name, (chrom, x, fv) in decoded.items():
        p         = fv / total_f
        lo        = cum
        cum      += p
        cum_data[name] = (chrom, x, fv, p, lo, cum)
        print(f"  {name:<12}  {str(chrom):<16}  {x:>4}  {fv:>6}  "
              f"{fv}/{total_f}={p:.4f}  {lo:.4f} - {cum:.4f}")

    print(f"\n  Roulette ranges:")
    for name, (_, _, _, _, lo, hi) in cum_data.items():
        print(f"    {name}: [{lo:.4f}, {hi:.4f})")

    # STEP 3 : Select parents using r1-r4
    print(f"\n STEP 3: Parent selection using r1=0.12, r2=0.47, r3=0.68, r4=0.91 \n")

    spins  = [0.12, 0.47, 0.68, 0.91]
    labels = ['r1', 'r2', 'r3', 'r4']
    selected = []

    for spin, label in zip(spins, labels):
        for name, (chrom, x, fv, p, lo, hi) in cum_data.items():
            if lo < spin <= hi or (lo == 0.0 and spin <= hi):
                print(f"  {label}={spin} -> falls in [{lo:.4f}, {hi:.4f}) "
                      f"-> selects {name} = {chrom}  (x={x}, f={fv})")
                selected.append((name, chrom, x, fv))
                break

    print(f"\n  Parent Pair 1 : {selected[0][0]} {selected[0][1]} "
          f"x {selected[1][0]} {selected[1][1]}")
    print(f"  Parent Pair 2 : {selected[2][0]} {selected[2][1]} "
          f"x {selected[3][0]} {selected[3][1]}")

    # STEP 4 : Single-point crossover at position 2
    print(f"\n STEP 4: Single-point crossover at position 2 (after bit index 2)")
    print(f"\n  Rule: O_odd  = Pa[0:2] + Pb[2:4]")
    print(f"        O_even = Pb[0:2] + Pa[2:4]\n")

    offspring = []
    for pair_num in range(2):
        pa_name, pa, pa_x, pa_f = selected[pair_num * 2]
        pb_name, pb, pb_x, pb_f = selected[pair_num * 2 + 1]
        pt  = 2
        o1  = pa[:pt] + pb[pt:]
        o2  = pb[:pt] + pa[pt:]
        x1  = o1[0]*8 + o1[1]*4 + o1[2]*2 + o1[3]*1
        x2  = o2[0]*8 + o2[1]*4 + o2[2]*2 + o2[3]*1
        f1  = -(x1**2) + 14*x1 + 5
        f2  = -(x2**2) + 14*x2 + 5
        n1  = f"O{pair_num*2+1}"
        n2  = f"O{pair_num*2+2}"

        print(f"  Pair {pair_num+1}: {pa_name} x {pb_name}")
        print(f"    {pa_name} = {pa}  -> bits[0:2]={pa[:pt]}  bits[2:4]={pa[pt:]}")
        print(f"    {pb_name} = {pb}  -> bits[0:2]={pb[:pt]}  bits[2:4]={pb[pt:]}")
        print(f"    {n1} = {pa[:pt]} + {pb[pt:]} = {o1}  "
              f"-> x={x1}, f={f1}" + (" [GLOBAL MAX]" if x1 == 7 else ""))
        print(f"    {n2} = {pb[:pt]} + {pa[pt:]} = {o2}  "
              f"-> x={x2}, f={f2}" + (" [GLOBAL MAX]" if x2 == 7 else ""))
        print()
        offspring += [(n1, o1, x1, f1), (n2, o2, x2, f2)]

    print(f"  All offspring after crossover:")
    print(f"  {'Name':<8}  {'Chromosome':<20}  {'x':>4}  {'f(x)':>6}  {'Note'}")
    for name, chrom, x, fv in offspring:
        note = "[GLOBAL MAX]" if x == 7 else ""
        print(f"  {name:<8}  {str(chrom):<20}  {x:>4}  {fv:>6}  {note}")

    # STEP 5 : Bit-flip mutation applied to O1
    print(f"\n STEP 5: Bit-flip mutation on O1, p_m=0.1 ")
    print(f"            per-bit random values = [0.08, 0.43, 0.91, 0.05]\n")
    print(f"  Rule: flip bit i if per_bit_r[i] < p_m (i.e. < 0.1)\n")

    o1_name, o1_chrom, o1_x, o1_f = offspring[0]
    per_bit_r = [0.08, 0.43, 0.91, 0.05]
    p_m       = 0.1

    print(f"  Original O1 = {o1_chrom}  (x={o1_x}, f={o1_f})\n")
    print(f"  {'Bit i':>6}  {'Value':>6}  {'Random r':>10}  {'r < 0.1?':>10}  "
          f"{'Action':>8}  {'Result':>8}")

    mutated = []
    for i, (bit, r) in enumerate(zip(o1_chrom, per_bit_r)):
        flip   = r < p_m
        action = "FLIP" if flip else "keep"
        result = 1 - bit if flip else bit
        mutated.append(result)
        print(f"  {i:>6}  {bit:>6}  {r:>10}  {'YES' if flip else 'no':>10}  "
              f"{action:>8}  {result:>8}")

    mut_x = mutated[0]*8 + mutated[1]*4 + mutated[2]*2 + mutated[3]*1
    mut_f = -(mut_x**2) + 14*mut_x + 5
    print(f"\n  Original O1  = {o1_chrom}  -> x={o1_x},  f={o1_f}")
    print(f"  Mutated  O1' = {mutated}  -> x={mut_x},  f={mut_f}"
          + (" [GLOBAL MAX]" if mut_x == 7 else ""))
    flipped_bits = [i for i, (a, b) in enumerate(zip(o1_chrom, mutated)) if a != b]
    print(f"  Bits flipped : {flipped_bits}  (r < 0.1 at those positions)")

    # STEP 6 : New generation summary
    print(f"\n STEP 6: New Generation Summary \n")

    _, o2_chrom, o2_x, o2_f = offspring[1]
    _, o3_chrom, o3_x, o3_f = offspring[2]
    _, o4_chrom, o4_x, o4_f = offspring[3]

    new_gen = [
        ("O1' (mutated)", mutated, mut_x, mut_f),
        ("O2",            o2_chrom, o2_x, o2_f),
        ("O3",            o3_chrom, o3_x, o3_f),
        ("O4",            o4_chrom, o4_x, o4_f),
    ]

    print(f"  {'Individual':<18}  {'Chromosome':<20}  {'x':>4}  {'f(x)':>6}  Note")
    print(f"  {'-'*65}")
    for name, chrom, x, fv in new_gen:
        note = "[GLOBAL MAX]" if x == 7 else ""
        print(f"  {name:<18}  {str(chrom):<20}  {x:>4}  {fv:>6}  {note}")

    orig_avg = sum(d[2] for d in decoded.values()) / 4
    new_avg  = sum(fv for _, _, _, fv in new_gen) / 4
    orig_best = max(d[2] for d in decoded.values())
    new_best  = max(fv for _, _, _, fv in new_gen)
    print(f"\n  Original population : avg f = {orig_avg:.2f},  best f = {orig_best}")
    print(f"  New generation      : avg f = {new_avg:.2f},  best f = {new_best}")

In [43]:
main_part_d()

PART (d) - Manual Trace: One Complete GA Generation
           No code-based GA calls -- all arithmetic shown explicitly

Starting population and given random numbers:
  P1 = [0,1,1,0]    r1 = 0.12  (selection spin 1)
  P2 = [1,0,0,1]    r2 = 0.47  (selection spin 2)
  P3 = [1,1,0,0]    r3 = 0.68  (selection spin 3)
  P4 = [0,0,1,1]    r4 = 0.91  (selection spin 4)
  Crossover point = 2 (after bit index 2)
  Mutation on O1 : per-bit random values = [0.08, 0.43, 0.91, 0.05], p_m = 0.1

 STEP 1: Decode chromosomes and compute f(x) = -x^2 + 14x + 5 

  P1 = [0, 1, 1, 0]
       decode  : 0*8 + 1*4 + 1*2 + 0*1 = 6
       f(x)    : -6^2 + 14*6 + 5 = -36 + 84 + 5 = 53
  P2 = [1, 0, 0, 1]
       decode  : 1*8 + 0*4 + 0*2 + 1*1 = 9
       f(x)    : -9^2 + 14*9 + 5 = -81 + 126 + 5 = 50
  P3 = [1, 1, 0, 0]
       decode  : 1*8 + 1*4 + 0*2 + 0*1 = 12
       f(x)    : -12^2 + 14*12 + 5 = -144 + 168 + 5 = 29
  P4 = [0, 0, 1, 1]
       decode  : 0*8 + 0*4 + 1*2 + 1*1 = 3
       f(x)    : -3^2 + 14*3 

In [44]:
print(f"""
Analysis:

  Selection step (r1=0.12, r2=0.47, r3=0.68, r4=0.91):
    r1=0.12 selects P1 [0,1,1,0] (range [0, 0.3118)) -- the highest-fitness
    individual (f=53) is selected, confirming roulette wheel's proportionate bias.
    r2=0.47 selects P2 (range [0.3118, 0.6059)). r3=0.68 selects P3 (f=29) --
    even the weakest individual can be selected since it still has non-zero
    probability. r4=0.91 selects P4 (range [0.7765, 1.000]).

  Crossover step (point=2):
    Pair 1 (P1 x P2): O1=[0,1,0,1] (x=5, f=50), O2=[1,0,1,0] (x=10, f=45).
    Both offspring are NEW chromosomes absent from the original population,
    demonstrating how crossover explores new regions of the search space by
    recombining building blocks [0,1] from P1 with [0,1] from P2.
    Pair 2 (P3 x P4): O3=[1,1,1,1] (x=15, f=-10), O4=[0,0,0,0] (x=0, f=5).
    These are the two extreme x values with very low fitness, showing that
    recombining lower-fitness parents can produce poor offspring.

  Mutation step (O1, p_m=0.1):
    Bits 0 and 3 are flipped (random values 0.08 and 0.05 are both < 0.1).
    O1=[0,1,0,1] (f=50) -> O1'=[1,1,0,0] (x=12, f=29). Mutation is harmful
    here -- it worsens O1. This is expected: mutation is occasionally
    disruptive by design. Over many generations, only the improvements
    survive selection pressure, so the long-run effect is beneficial.
    The two flipped bits change x from 5 to 12, moving far from the optimum x=7.
""")



Analysis:

  Selection step (r1=0.12, r2=0.47, r3=0.68, r4=0.91):
    r1=0.12 selects P1 [0,1,1,0] (range [0, 0.3118)) -- the highest-fitness
    individual (f=53) is selected, confirming roulette wheel's proportionate bias.
    r2=0.47 selects P2 (range [0.3118, 0.6059)). r3=0.68 selects P3 (f=29) --
    even the weakest individual can be selected since it still has non-zero
    probability. r4=0.91 selects P4 (range [0.7765, 1.000]).

  Crossover step (point=2):
    Pair 1 (P1 x P2): O1=[0,1,0,1] (x=5, f=50), O2=[1,0,1,0] (x=10, f=45).
    Both offspring are NEW chromosomes absent from the original population,
    demonstrating how crossover explores new regions of the search space by
    recombining building blocks [0,1] from P1 with [0,1] from P2.
    Pair 2 (P3 x P4): O3=[1,1,1,1] (x=15, f=-10), O4=[0,0,0,0] (x=0, f=5).
    These are the two extreme x values with very low fitness, showing that
    recombining lower-fitness parents can produce poor offspring.

  Mutation step (O1,

In [45]:
#  RUN ALL PARTS

if __name__ == "__main__":
    main_part_a()
    main_part_b()
    main_part_c_experiment1()
    main_part_c_experiment2()
    main_part_d()

PART (a) - Core GA Components: decode, fitness, roulette_select,
           single_point_crossover, mutate

Objective function  : f(x) = -x^2 + 14x + 5,   x in {0, 1, ..., 15}
Chromosome          : 4-bit binary list, MSB first
Global maximum      : x = 7,  f(7) = -(49) + 98 + 5 = 54
Test chromosomes    : [0,1,1,0], [1,0,0,1], [1,1,0,0], [0,0,1,1]

--- TEST 1: decode(chromosome) ---
  Rule: MSB first -> value = b3*8 + b2*4 + b1*2 + b0*1

  Chromosome            Working                                      x
  ----------------------------------------------------------------------
  [0, 1, 1, 0]          0*8 + 1*4 + 1*2 + 0*1 = 6                    6  OK
  [1, 0, 0, 1]          1*8 + 0*4 + 0*2 + 1*1 = 9                    9  OK
  [1, 1, 0, 0]          1*8 + 1*4 + 0*2 + 0*1 = 12                  12  OK
  [0, 0, 1, 1]          0*8 + 0*4 + 1*2 + 1*1 = 3                    3  OK

--- TEST 2: fitness(chromosome)  [f(x) = -x^2 + 14x + 5] ---

  Chromosome               x  Working               

# Q5

In [46]:
import random

In [47]:
COURSES      = 6        # C1 .. C6
ROOMS        = 3        # R1 R2 R3  (indices 0,1,2)
SLOTS        = 4        # T1 T2 T3 T4  (indices 0,1,2,3)
TOTAL_CELLS  = ROOMS * SLOTS   # 12 possible (room, slot) pairs

In [48]:
# CORE REPRESENTATION HELPERS (used across all parts)

def random_chromosome():
    """
    Generate a random chromosome: list of 6 (room_idx, slot_idx) tuples.
    room_idx in {0,1,2},  slot_idx in {0,1,2,3}.
    No validity constraint enforced -- conflicts are penalised by fitness.
    """
    return [(random.randint(0, ROOMS - 1), random.randint(0, SLOTS - 1))
            for _ in range(COURSES)]

In [49]:
def count_conflicts(chromosome):
    """
    Count pairs of courses assigned to the same room AND same time slot.
    Each such pair = 1 conflict.  A conflict-free schedule scores 0.

    Method: compare every pair (i, j) with i < j.
            If chromosome[i] == chromosome[j] -> same room + same slot -> conflict.
    """
    conflicts = 0
    for i in range(COURSES):
        for j in range(i + 1, COURSES):
            if chromosome[i] == chromosome[j]:
                conflicts += 1
    return conflicts

In [50]:
def fitness(chromosome):
    """
    f = 100 - (10 * conflicts).
    Conflict-free schedule -> 100.  Each conflict deducts 10 points.
    """
    return 100 - 10 * count_conflicts(chromosome)

In [51]:
def chromosome_str(chrom):
    """Human-readable chromosome: C1:(R1,T1) C2:(R2,T3) ..."""
    parts = []
    for i, (r, s) in enumerate(chrom):
        parts.append(f"C{i+1}:(R{r+1},T{s+1})")
    return "  ".join(parts)

In [52]:
#  PART (a) - Representation, count_conflicts, fitness

def main_part_a():
    print("PART (a) - Representation, count_conflicts, fitness, random_chromosome")

    # Demonstrate count_conflicts on known cases
    print(" Demonstration: count_conflicts on known chromosomes \n")

    demo_cases = [
        # (chromosome, description)
        ([(0,0),(1,0),(2,0),(0,1),(1,1),(2,1)],
         "All different cells -- should be 0 conflicts"),
        ([(0,0),(0,0),(1,1),(2,2),(0,3),(1,2)],
         "C1 and C2 share (R1,T1) -- should be 1 conflict"),
        ([(0,0),(0,0),(0,0),(1,1),(1,1),(2,2)],
         "C1=C2=C3 share (R1,T1), C4=C5 share (R2,T2) -- should be 4 pairs"),
        ([(0,0),(0,0),(0,0),(0,0),(0,0),(0,0)],
         "All six in same cell -- C(6,2)=15 conflict pairs"),
    ]

    print(f"  {'Description':<52}  {'Conflicts':>9}  {'f':>5}")
    for chrom, desc in demo_cases:
        c = count_conflicts(chrom)
        f = fitness(chrom)
        ok = " [CONFLICT-FREE]" if c == 0 else ""
        print(f"  {desc:<52}  {c:>9}  {f:>5}{ok}")

    print()

    # Generate and print 5 random chromosomes
    print(" 5 Random Chromosomes (seed=42) \n")

    print(f"  {'#':<4}  {'Chromosome':<60}  {'Conflicts':>9}  {'f':>5}")
    for i in range(1, 6):
        chrom = random_chromosome()
        c     = count_conflicts(chrom)
        f     = fitness(chrom)
        note  = " [CONFLICT-FREE]" if c == 0 else ""
        print(f"  {i:<4}  {chromosome_str(chrom):<60}  {c:>9}  {f:>5}{note}")

In [53]:
main_part_a()

PART (a) - Representation, count_conflicts, fitness, random_chromosome
 Demonstration: count_conflicts on known chromosomes 

  Description                                           Conflicts      f
  All different cells -- should be 0 conflicts                  0    100 [CONFLICT-FREE]
  C1 and C2 share (R1,T1) -- should be 1 conflict               1     90
  C1=C2=C3 share (R1,T1), C4=C5 share (R2,T2) -- should be 4 pairs          4     60
  All six in same cell -- C(6,2)=15 conflict pairs             15    -50

 5 Random Chromosomes (seed=42) 

  #     Chromosome                                                    Conflicts      f
  1     C1:(R2,T3)  C2:(R3,T3)  C3:(R1,T4)  C4:(R2,T2)  C5:(R1,T4)  C6:(R1,T4)          3     70
  2     C1:(R1,T1)  C2:(R2,T2)  C3:(R3,T4)  C4:(R1,T3)  C5:(R1,T4)  C6:(R1,T2)          0    100 [CONFLICT-FREE]
  3     C1:(R1,T2)  C2:(R1,T2)  C3:(R1,T4)  C4:(R2,T1)  C5:(R1,T3)  C6:(R2,T2)          1     90
  4     C1:(R3,T4)  C2:(R2,T3)  C3:(R2,T3)  C4:(R1,T

In [54]:
print(f"""
Analysis:
  The total number of possible chromosomes is (ROOMS * SLOTS)^COURSES
  = 12^6 = 2,985,984. Of these, the number of conflict-free schedules
  is P(12,6) = 12*11*10*9*8*7 = 665,280 (each course gets a distinct
  cell). The fraction of valid schedules is 665,280 / 2,985,984 = 22.3%.
  With such a relatively high valid fraction, a GA that penalises
  conflicts and uses repair has a good chance of converging to f=100.
  The 5 random chromosomes above typically show 1-3 conflicts because
  with 6 courses placed into 12 cells, collisions are moderately likely
  (birthday-problem analogy: P(at least one collision) ~ 67%).
""")


Analysis:
  The total number of possible chromosomes is (ROOMS * SLOTS)^COURSES
  = 12^6 = 2,985,984. Of these, the number of conflict-free schedules
  is P(12,6) = 12*11*10*9*8*7 = 665,280 (each course gets a distinct
  cell). The fraction of valid schedules is 665,280 / 2,985,984 = 22.3%.
  With such a relatively high valid fraction, a GA that penalises
  conflicts and uses repair has a good chance of converging to f=100.
  The 5 random chromosomes above typically show 1-3 conflicts because
  with 6 courses placed into 12 cells, collisions are moderately likely
  (birthday-problem analogy: P(at least one collision) ~ 67%).



In [55]:
#  PART (b) - Crossover, repair, mutate

def crossover(p1, p2, point):
    """
    Standard single-point crossover at gene index `point`.
    offspring1 = p1[:point] + p2[point:]
    offspring2 = p2[:point] + p1[point:]
    May produce conflicting offspring -- repair() handles that.
    point must be in [1, COURSES-1].
    """
    o1 = p1[:point] + p2[point:]
    o2 = p2[:point] + p1[point:]
    return o1, o2

In [56]:
def repair(chromosome):
    """
    Post-crossover repair: find all conflicting courses and reassign them.

    Algorithm:
      1. Build a dict mapping each (room,slot) cell to the list of course
         indices that occupy it.
      2. For any cell occupied by more than one course, keep the FIRST
         course in that cell (it stays) and mark the rest as conflicting.
      3. For each conflicting course, try to find a conflict-free cell
         (a (room,slot) pair not already occupied by another course).
      4. If a free cell exists, assign the course there (random choice).
         Otherwise, assign randomly to any cell (unavoidable conflict).
    Returns a NEW chromosome (does not modify in place).
    """
    repaired = chromosome[:]

    # Build occupancy map: cell -> [course indices]
    occupancy = {}
    for i, cell in enumerate(repaired):
        occupancy.setdefault(cell, []).append(i)

    # Identify which course indices are "losers" in each collision
    conflicting = []
    for cell, indices in occupancy.items():
        if len(indices) > 1:
            conflicting.extend(indices[1:])   # keep indices[0], rest must move

    # All currently occupied cells (after winners are settled)
    occupied_cells = set(repaired[i] for i in range(COURSES)
                         if i not in conflicting)

    # All possible (room,slot) cells
    all_cells = [(r, s) for r in range(ROOMS) for s in range(SLOTS)]

    # Reassign each conflicting course
    for ci in conflicting:
        free_cells = [c for c in all_cells if c not in occupied_cells]
        if free_cells:
            chosen = random.choice(free_cells)
        else:
            chosen = random.choice(all_cells)   # no free cell: random fallback
        repaired[ci]  = chosen
        occupied_cells.add(chosen)

    return repaired

In [57]:
def mutate(chromosome, p_m):
    """
    Per-gene mutation: with probability p_m, reassign that course to a
    random (room_idx, slot_idx) pair.  Returns a NEW chromosome.
    """
    return [(random.randint(0, ROOMS - 1), random.randint(0, SLOTS - 1))
            if random.random() < p_m else gene
            for gene in chromosome]

In [62]:
def main_part_b():
    print("PART (b) - Crossover, repair, mutate (with demonstrations)")

    print("""
Three operators implemented:
  crossover(p1, p2, point) : single-point crossover; offspring may conflict.
  repair(chromosome)       : detects conflicting courses, reassigns to free cells.
  mutate(chromosome, p_m)  : per-gene random reassignment at rate p_m.
""")

    # DEMONSTRATION 1 : crossover creates a conflict
    print(" Demonstration 1: How crossover creates a conflict \n")

    # Design two parents with NO conflicts individually.
    # Parent A: C1-C6 each in a distinct cell, all different rooms/slots
    # Parent B: also conflict-free, but chosen so crossover will collide
    parent_a = [(0,0),(1,1),(2,2),(0,3),(1,2),(2,1)]   # 0 conflicts
    parent_b = [(2,0),(0,1),(1,2),(2,3),(0,2),(1,1)]   # 0 conflicts

    ca = count_conflicts(parent_a)
    cb = count_conflicts(parent_b)

    print(f"  Parent A : {chromosome_str(parent_a)}")
    print(f"             conflicts = {ca},  f = {fitness(parent_a)}")
    print(f"  Parent B : {chromosome_str(parent_b)}")
    print(f"             conflicts = {cb},  f = {fitness(parent_b)}")

    # Crossover at point 3 (first 3 genes from A, last 3 from B)
    point = 3
    o1, o2 = crossover(parent_a, parent_b, point)
    co1 = count_conflicts(o1)
    co2 = count_conflicts(o2)

    print(f"\n  Crossover at point = {point}:")
    print(f"    Offspring1 = A[:3] + B[3:] = {parent_a[:point]} + {parent_b[point:]}")
    print(f"    Offspring1 : {chromosome_str(o1)}")
    print(f"               conflicts = {co1},  f = {fitness(o1)}")

    # Explain the specific collision in o1
    cell_map = {}
    for i, cell in enumerate(o1):
        cell_map.setdefault(cell, []).append(f"C{i+1}")
    for cell, courses in cell_map.items():
        if len(courses) > 1:
            print(f"    -> Conflict: {' and '.join(courses)} both assigned "
                  f"to (R{cell[0]+1},T{cell[1]+1})")

    print(f"\n    Offspring2 = B[:3] + A[3:] = {parent_b[:point]} + {parent_a[point:]}")
    print(f"    Offspring2 : {chromosome_str(o2)}")
    print(f"               conflicts = {co2},  f = {fitness(o2)}")

    cell_map2 = {}
    for i, cell in enumerate(o2):
        cell_map2.setdefault(cell, []).append(f"C{i+1}")
    for cell, courses in cell_map2.items():
        if len(courses) > 1:
            print(f"    -> Conflict: {' and '.join(courses)} both assigned "
                  f"to (R{cell[0]+1},T{cell[1]+1})")

    # DEMONSTRATION 2 : repair() fixes a conflicting chromosome
    print(" Demonstration 2: repair() fixes a conflicting chromosome \n")
    random.seed(7)

    # Manually create a chromosome with known conflicts
    # C1 and C3 both in (R1,T1); C4 and C6 both in (R2,T2)
    conflicted = [(0,0),(1,2),(0,0),(1,1),(2,3),(1,1)]
    c_before   = count_conflicts(conflicted)
    f_before   = fitness(conflicted)

    print(f"  Conflicted chromosome (manually constructed):")
    print(f"  {chromosome_str(conflicted)}")
    print(f"  conflicts before repair = {c_before},  f = {f_before}")

    # Show which pairs conflict
    for i in range(COURSES):
        for j in range(i+1, COURSES):
            if conflicted[i] == conflicted[j]:
                r, s = conflicted[i]
                print(f"  -> Conflict: C{i+1} and C{j+1} share (R{r+1},T{s+1})")

    repaired   = repair(conflicted)
    c_after    = count_conflicts(repaired)
    f_after    = fitness(repaired)

    print(f"\n  After repair():")
    print(f"  {chromosome_str(repaired)}")
    print(f"  conflicts after repair  = {c_after},  f = {f_after}")
    if c_after == 0:
        print(f"  Result: CONFLICT-FREE schedule achieved by repair!")
    else:
        print(f"  Result: conflicts reduced from {c_before} to {c_after}.")

    print(f"""
  How repair() works:
    1. Build an occupancy map: each (room,slot) cell -> list of course indices.
    2. In any cell with multiple courses, keep the first (index 0) and flag
       the rest as "conflicting" -- they must be moved.
    3. For each conflicting course, collect all (room,slot) cells not currently
       occupied by any settled course.  Assign a random free cell.
    4. If no free cell exists (all 12 cells taken), fall back to a random cell.
    repair() is applied after EVERY crossover in the scheduling GA to ensure
    offspring are at least partially feasible before selection.
""")

    # DEMONSTRATION 3 : mutate()

    print(" Demonstration 3: mutate(chromosome, p_m) \n")

    base = [(0,0),(1,1),(2,2),(0,3),(1,2),(2,1)]
    print(f"  Base chromosome : {chromosome_str(base)}")
    print(f"  conflicts = {count_conflicts(base)},  f = {fitness(base)}\n")
    print(f"  {'p_m':<6}  {'Mutated chromosome':<60}  {'Conflicts':>9}  {'f':>5}")
    for p_m in [0.0, 0.1, 0.5, 1.0]:
        random.seed(20)
        mut   = mutate(base, p_m)
        c_mut = count_conflicts(mut)
        f_mut = fitness(mut)
        changed = sum(1 for a, b in zip(base, mut) if a != b)
        print(f"  {p_m:<6}  {chromosome_str(mut):<60}  {c_mut:>9}  {f_mut:>5}  ({changed} gene(s) changed)")

In [63]:
main_part_b()

PART (b) - Crossover, repair, mutate (with demonstrations)

Three operators implemented:
  crossover(p1, p2, point) : single-point crossover; offspring may conflict.
  repair(chromosome)       : detects conflicting courses, reassigns to free cells.
  mutate(chromosome, p_m)  : per-gene random reassignment at rate p_m.

 Demonstration 1: How crossover creates a conflict 

  Parent A : C1:(R1,T1)  C2:(R2,T2)  C3:(R3,T3)  C4:(R1,T4)  C5:(R2,T3)  C6:(R3,T2)
             conflicts = 0,  f = 100
  Parent B : C1:(R3,T1)  C2:(R1,T2)  C3:(R2,T3)  C4:(R3,T4)  C5:(R1,T3)  C6:(R2,T2)
             conflicts = 0,  f = 100

  Crossover at point = 3:
    Offspring1 = A[:3] + B[3:] = [(0, 0), (1, 1), (2, 2)] + [(2, 3), (0, 2), (1, 1)]
    Offspring1 : C1:(R1,T1)  C2:(R2,T2)  C3:(R3,T3)  C4:(R3,T4)  C5:(R1,T3)  C6:(R2,T2)
               conflicts = 1,  f = 90
    -> Conflict: C2 and C6 both assigned to (R2,T2)

    Offspring2 = B[:3] + A[3:] = [(2, 0), (0, 1), (1, 2)] + [(0, 3), (1, 2), (2, 1)]
    Offs

In [64]:
print(f"""
  Why crossover introduces conflicts:
    Parent A's left genes and Parent B's right genes are designed to be
    individually conflict-free but they were optimised independently.
    When the prefix of A is joined with the suffix of B, the same
    (room,slot) cell can appear twice -- once inherited from A and once
    from B -- because neither parent "knew" what the other would contribute.
    This is the fundamental tension in constraint-satisfaction GAs: operators
    that work on independent gene blocks cannot guarantee global feasibility.
""")


  Why crossover introduces conflicts:
    Parent A's left genes and Parent B's right genes are designed to be
    individually conflict-free but they were optimised independently.
    When the prefix of A is joined with the suffix of B, the same
    (room,slot) cell can appear twice -- once inherited from A and once
    from B -- because neither parent "knew" what the other would contribute.
    This is the fundamental tension in constraint-satisfaction GAs: operators
    that work on independent gene blocks cannot guarantee global feasibility.



In [65]:
#  PART (c) - Full Scheduling GA

def tournament_select(population, k=2):
    """
    Tournament selection: randomly pick k individuals, return the fittest.
    k=2 (binary tournament) as specified.
    """
    contestants = random.sample(population, k)
    return max(contestants, key=fitness)

In [66]:
def run_scheduling_ga(pop_size, generations, p_m, verbose=True):
    """
    Full Genetic Algorithm for the Course Scheduling Problem.

    Steps each generation:
      1. Evaluate fitness of every individual.
      2. Record best chromosome and fitness.
      3. Build next generation:
           a) Tournament selection (size=2) for two parents.
           b) Single-point crossover at a random gene index [1, COURSES-1].
           c) repair() applied to both offspring.
           d) mutate() applied to both offspring.
      4. Repeat until pop_size offspring generated.
    If a conflict-free chromosome (f=100) is found, report the generation.

    Returns (best_chromosome, best_fitness, fitness_history)
    """
    # Initialise population and immediately repair all chromosomes
    population     = [repair(random_chromosome()) for _ in range(pop_size)]
    fitness_history = []
    solution_gen   = None
    best_ever      = None
    best_ever_f    = -1

    for gen in range(1, generations + 1):

        # Evaluate
        scored = [(chrom, fitness(chrom)) for chrom in population]
        scored.sort(key=lambda t: t[1], reverse=True)

        gen_best_chrom, gen_best_f = scored[0]
        fitness_history.append(gen_best_f)

        if gen_best_f > best_ever_f:
            best_ever_f    = gen_best_f
            best_ever      = gen_best_chrom[:]

        # Detect solution
        if gen_best_f == 100 and solution_gen is None:
            solution_gen = gen
            if verbose:
                print(f"  Solution found at generation {gen}: {gen_best_chrom}")

        if verbose and (gen <= 5 or gen % 10 == 0 or gen == generations):
            avg_f = sum(f for _, f in scored) / pop_size
            print(f"  Gen {gen:>3} | best f = {gen_best_f:>4} | avg f = {avg_f:>6.2f} | "
                  f"best chrom = {gen_best_chrom}")

        # Build next generation
        next_pop = []
        while len(next_pop) < pop_size:
            p1 = tournament_select(population)
            p2 = tournament_select(population)

            point = random.randint(1, COURSES - 1)
            o1, o2 = crossover(p1, p2, point)

            # Repair then mutate
            o1 = mutate(repair(o1), p_m)
            o2 = mutate(repair(o2), p_m)

            next_pop.append(o1)
            if len(next_pop) < pop_size:
                next_pop.append(o2)

        population = next_pop

    return best_ever, best_ever_f, fitness_history, solution_gen

In [67]:
def main_part_c():
    print("\n" + "=" * 75)
    print("PART (c) - Full Scheduling GA")
    print("           pop_size=20, generations=50, p_m=0.1")
    print("=" * 75)

    print("""
Algorithm:
  Selection  : Tournament (size=2) -- pick 2 random individuals, keep fitter one.
  Crossover  : Single-point at random gene index in [1, 5].
  Repair     : Applied to EVERY offspring immediately after crossover.
  Mutation   : Per-gene random reassignment at rate p_m=0.1.
  Population : Initialised with repair() so all starting chromosomes are
               as feasible as possible before evolution begins.
""")

    random.seed(42)
    print("--- GA Run: pop_size=20, generations=50, p_m=0.1 (seed=42) ---\n")
    print(f"  {'Gen':>4}  {'Best f':>7}  {'Avg f':>7}  Best Chromosome")

    best_chrom, best_f, history, sol_gen = run_scheduling_ga(
        pop_size=20, generations=50, p_m=0.1, verbose=True
    )

    # Best schedule found
    print(f"\n Best Schedule Found \n")
    print(f"  Fitness          : {best_f}")
    print(f"  Conflicts        : {count_conflicts(best_chrom)}")
    print(f"  Chromosome       : {best_chrom}")
    if sol_gen:
        print(f"  Solution at gen  : {sol_gen}")
    else:
        print(f"  Solution found   : No (0-conflict schedule not achieved in this run)")

    print(f"\n  Course assignments:")
    print(f"  {'Course':<10}  {'Room':>5}  {'Time Slot':>10}  {'Note'}")
    for i, (r, s) in enumerate(best_chrom):
        print(f"  C{i+1:<9}  R{r+1:>5}  T{s+1:>10}")

    # Check for any remaining conflicts
    cell_map = {}
    for i, cell in enumerate(best_chrom):
        cell_map.setdefault(cell, []).append(f"C{i+1}")
    conflict_pairs = [(cell, courses) for cell, courses in cell_map.items()
                      if len(courses) > 1]
    if conflict_pairs:
        print(f"\n  Remaining conflicts:")
        for (r, s), courses in conflict_pairs:
            print(f"    {' and '.join(courses)} share (R{r+1},T{s+1})")
    else:
        print(f"\n  No conflicts -- schedule is fully feasible!")

    # Fitness progression
    print(f"\n Fitness Progression Per Generation \n")
    print(f"  {'Gen':>4}  {'Best f':>7}  {'Bar'}")
    for gen, f_val in enumerate(history, 1):
        if gen <= 10 or gen % 5 == 0:
            bar = "#" * (f_val // 5)
            print(f"  {gen:>4}  {f_val:>7}  {bar}")

    # --- Run 10 independent trials to assess reliability ---
    print(f"\n Reliability Test: 10 Independent Runs \n")
    print(f"  {'Run':>4}  {'Best f':>7}  {'Conflicts':>10}  {'Sol Gen':>8}  {'Found?':>7}")
    successes = 0
    sol_gens  = []
    for run in range(1, 11):
        random.seed(run * 17)
        bc, bf, _, sg = run_scheduling_ga(20, 50, 0.1, verbose=False)
        cc  = count_conflicts(bc)
        found = "YES" if bf == 100 else "No"
        sg_str = str(sg) if sg else "N/A"
        if bf == 100:
            successes += 1
            sol_gens.append(sg)
        print(f"  {run:>4}  {bf:>7}  {cc:>10}  {sg_str:>8}  {found:>7}")

    avg_sg = sum(sol_gens) / len(sol_gens) if sol_gens else None
    print(f"\n  Conflict-free solutions found : {successes}/10")
    if avg_sg:
        print(f"  Avg generation to solution   : {avg_sg:.1f}")
    else:
        print(f"  Avg generation to solution   : N/A (no solutions found)")

In [68]:
main_part_c()


PART (c) - Full Scheduling GA
           pop_size=20, generations=50, p_m=0.1

Algorithm:
  Selection  : Tournament (size=2) -- pick 2 random individuals, keep fitter one.
  Crossover  : Single-point at random gene index in [1, 5].
  Repair     : Applied to EVERY offspring immediately after crossover.
  Mutation   : Per-gene random reassignment at rate p_m=0.1.
  Population : Initialised with repair() so all starting chromosomes are
               as feasible as possible before evolution begins.

--- GA Run: pop_size=20, generations=50, p_m=0.1 (seed=42) ---

   Gen   Best f    Avg f  Best Chromosome
  Solution found at generation 1: [(2, 0), (0, 2), (0, 1), (0, 0), (0, 3), (2, 3)]
  Gen   1 | best f =  100 | avg f = 100.00 | best chrom = [(2, 0), (0, 2), (0, 1), (0, 0), (0, 3), (2, 3)]
  Gen   2 | best f =  100 | avg f =  99.00 | best chrom = [(0, 0), (0, 3), (1, 2), (1, 0), (2, 0), (2, 3)]
  Gen   3 | best f =  100 | avg f =  97.50 | best chrom = [(2, 3), (0, 0), (0, 3), (2, 1), (1,

In [69]:
#  PART (d) - Written Analysis

def main_part_d():
    print("PART (d) - Written Analysis")

    # Compute the exact numbers for the analysis
    import math
    total_chromosomes  = TOTAL_CELLS ** COURSES
    # Conflict-free: choose 6 DISTINCT cells from 12 -- ordered (each course is distinct)
    valid_schedules    = math.perm(TOTAL_CELLS, COURSES)   # P(12,6) = 12*11*10*9*8*7
    valid_fraction     = valid_schedules / total_chromosomes

    print(f"""
--- Part (d.1): Did the GA consistently find a conflict-free schedule? ---

  Total possible chromosomes  : {ROOMS}^COURSES * {SLOTS}^COURSES
                              = 12^{COURSES} = {total_chromosomes:,}
  Conflict-free schedules     : P(12,{COURSES}) = 12*11*10*9*8*7 = {valid_schedules:,}
  Fraction of valid schedules : {valid_schedules:,} / {total_chromosomes:,}
                              = {valid_fraction:.4f}  ({valid_fraction*100:.1f}%)

  From the 10 independent runs above, the GA found conflict-free schedules
  in the majority of trials. When it does NOT find a solution, the cause is
  premature convergence: once the population clusters around a near-optimal
  but conflicted chromosome, mutation at p_m=0.1 may not generate the exact
  gene change needed to move the last conflicting course to a free cell.
  The constraint that makes this hardest is the room-slot UNIQUENESS rule:
  with 6 courses competing for 12 distinct (room,slot) cells, the final
  assignment requires a perfect matching. As the GA converges, the repair
  operator must work within an increasingly constrained residual space,
  and with only 20 individuals in the population, genetic diversity
  sometimes collapses before the last collision is resolved.

  The valid-schedule fraction is {valid_fraction*100:.1f}% -- much higher than many
  combinatorial problems -- which is why the GA succeeds most of the time.
  If the problem had fewer cells (e.g. 3 slots instead of 4), the valid
  fraction would drop to P(9,6)/9^6 = 60,480/531,441 = 11.4%, making it
  significantly harder and increasing the failure rate.

--- Part (d.2): GA vs RRHC -- when is each preferable? ---

  TWO CONDITIONS WHERE THIS GA IS PREFERABLE TO RRHC:
  1. Large search space with combinatorial structure:
     The scheduling problem has 12^6 = 2,985,984 possible chromosomes.
     RRHC evaluates one random starting point per restart and climbs
     greedily -- it is blind to the population-level patterns that the
     GA exploits. The GA's crossover operator recombines PARTIAL solutions
     (e.g., a conflict-free assignment for C1-C3 from one parent, and
     a conflict-free assignment for C4-C6 from another), assembling
     feasible sub-schedules into full solutions. RRHC cannot recombine
     information from different search trajectories.

  2. Dense constraint interaction between genes:
     In this problem, assigning one course's room and slot affects the
     feasibility of all other courses (any shared cell is a conflict).
     The GA's population maintains DIVERSE candidate solutions
     simultaneously, allowing crossover to find synergistic combinations.
     RRHC with a swap neighbourhood evaluates one candidate at a time;
     its local neighbourhood is blind to improvements that require
     simultaneously moving two or more courses.

  TWO CONDITIONS WHERE RRHC IS PREFERABLE TO THIS GA:
  1. Very small search space or when any feasible solution is acceptable:
     If the number of courses were 3 instead of 6, the valid-schedule
     fraction rises to P(12,3)/12^3 = 1320/1728 = 76.4%. In that case
     a random restart approach finds a conflict-free schedule almost
     immediately (within 1-2 restarts on average), while the GA wastes
     effort maintaining a population of 20 chromosomes, running selection,
     crossover, and mutation for 50 generations unnecessarily.

  2. When implementation simplicity and runtime speed matter most:
     RRHC for this problem requires only a random_chromosome() function
     and a simple greedy assignment step (assign courses to unused cells).
     The GA requires five components (selection, crossover, repair,
     mutation, population management) and runs pop_size * generations =
     20 * 50 = 1,000 evaluations. For problems where the constraint
     density is low (few conflicts in random chromosomes) or where a
     good-enough (not optimal) solution suffices, RRHC's simplicity and
     low overhead make it the practical choice. Population diversity --
     the GA's main advantage -- only pays off when the landscape has many
     local optima that RRHC gets trapped in, which does not occur when
     the feasible fraction is high and the landscape is smooth.
""")

In [70]:
main_part_d()

PART (d) - Written Analysis

--- Part (d.1): Did the GA consistently find a conflict-free schedule? ---

  Total possible chromosomes  : 3^COURSES * 4^COURSES
                              = 12^6 = 2,985,984
  Conflict-free schedules     : P(12,6) = 12*11*10*9*8*7 = 665,280
  Fraction of valid schedules : 665,280 / 2,985,984
                              = 0.2228  (22.3%)

  From the 10 independent runs above, the GA found conflict-free schedules
  in the majority of trials. When it does NOT find a solution, the cause is
  premature convergence: once the population clusters around a near-optimal
  but conflicted chromosome, mutation at p_m=0.1 may not generate the exact
  gene change needed to move the last conflicting course to a free cell.
  The constraint that makes this hardest is the room-slot UNIQUENESS rule:
  with 6 courses competing for 12 distinct (room,slot) cells, the final
  assignment requires a perfect matching. As the GA converges, the repair
  operator must work within

In [71]:
#  RUN ALL PARTS

if __name__ == "__main__":
    main_part_a()
    main_part_b()
    main_part_c()
    main_part_d()

PART (a) - Representation, count_conflicts, fitness, random_chromosome
 Demonstration: count_conflicts on known chromosomes 

  Description                                           Conflicts      f
  All different cells -- should be 0 conflicts                  0    100 [CONFLICT-FREE]
  C1 and C2 share (R1,T1) -- should be 1 conflict               1     90
  C1=C2=C3 share (R1,T1), C4=C5 share (R2,T2) -- should be 4 pairs          4     60
  All six in same cell -- C(6,2)=15 conflict pairs             15    -50

 5 Random Chromosomes (seed=42) 

  #     Chromosome                                                    Conflicts      f
  1     C1:(R3,T2)  C2:(R2,T4)  C3:(R3,T3)  C4:(R1,T4)  C5:(R3,T3)  C6:(R3,T1)          1     90
  2     C1:(R3,T1)  C2:(R3,T1)  C3:(R1,T2)  C4:(R3,T4)  C5:(R1,T2)  C6:(R2,T3)          2     80
  3     C1:(R1,T3)  C2:(R2,T1)  C3:(R3,T4)  C4:(R3,T1)  C5:(R1,T3)  C6:(R3,T3)          1     90
  4     C1:(R3,T3)  C2:(R3,T2)  C3:(R3,T3)  C4:(R2,T2)  C5:(R1,T2)  